# M.Sc. Project Notebook: Optimizing Retail Stock Levels Through AI-Powered Demand Prediction

This notebook documents the complete workflow for my final-year M.Sc. Data Science project. The project is designed around a practical retail question: **how can demand forecasting be used to improve stock planning decisions?**

Instead of stopping at prediction alone, the notebook connects forecasting results to reorder points, safety stock, simulated stock movement, and alert generation. That makes the project closer to a real decision-support workflow rather than a standalone machine learning exercise.

## 1. Project Context and Motivation

Retail businesses constantly face a balance problem. If they overestimate demand, they tie up capital in excess stock and increase holding cost. If they underestimate demand, they face stockouts, missed sales, and customer dissatisfaction.

For this reason, I framed the project in three linked stages:

1. prepare a clean and explainable demand dataset,
2. forecast short-term demand using more than one modeling approach,
3. convert those forecasts into inventory recommendations and stock alerts.

This structure matches the overall goal of optimizing retail stock levels through AI-powered demand prediction.

## 2. Objectives of the Work

The notebook aims to satisfy the academic requirements of the project in a structured way:

- perform data collection, merging, and cleaning,
- carry out exploratory data analysis,
- compare forecasting methods,
- include classical and AI-oriented forecasting approaches where the environment allows,
- design an inventory optimization layer,
- generate dashboard-ready outputs and alert-style summaries.

To keep the project manageable and interpretable, I focus on **store 1** and three product families:

- `BEVERAGES`
- `DAIRY`
- `GROCERY I`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'Dataset' / 'Corporacion_Favorita'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'store1_three_families_daily.csv'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
TABLES_DIR = REPORTS_DIR / 'tables'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('ggplot')

## 3. Raw Data Sources

The original dataset folder contains multiple files with different types of information. I did not want the project to depend only on a manually filtered CSV file, so I rebuilt the processed dataset from these raw sources.

The main files used are:

- `train.csv` for daily sales,
- `items.csv` for product family information,
- `stores.csv` for store metadata,
- `oil.csv` for oil price series,
- `holidays_events.csv` for event indicators,
- `transactions.csv` for daily store activity.

In [ ]:
raw_files = {
    'train.csv': RAW_DIR / 'train.csv',
    'items.csv': RAW_DIR / 'items.csv',
    'stores.csv': RAW_DIR / 'stores.csv',
    'oil.csv': RAW_DIR / 'oil.csv',
    'holidays_events.csv': RAW_DIR / 'holidays_events.csv',
    'transactions.csv': RAW_DIR / 'transactions.csv',
}

pd.DataFrame([
    {'file': name, 'exists': path.exists(), 'path': str(path)}
    for name, path in raw_files.items()
])

In [ ]:
pd.read_csv(raw_files['train.csv'], nrows=5)

### Why the Scope Was Reduced

The full Corporacion Favorita dataset is large and contains many stores and items. For a master's project, I believe it is better to build a clear, well-argued solution on a narrower slice of the data than to attempt a huge workflow that is hard to explain.

So I intentionally limited the analysis to one store and three product families. This still preserves meaningful demand variation while keeping the notebook readable and the evaluation manageable.

## 4. Loading the Processed Dataset

The processed dataset was generated by the project preprocessing script and saved to `data/processed/store1_three_families_daily.csv`. Each row represents one day, one store, and one product family.

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=['date'])
df = df.sort_values(['family', 'date']).reset_index(drop=True)
df.head()

In [ ]:
summary = pd.Series({
    'Rows': len(df),
    'Columns': len(df.columns),
    'Start date': df['date'].min().date(),
    'End date': df['date'].max().date(),
    'Stores': int(df['store_nbr'].nunique()),
    'Families': ', '.join(sorted(df['family'].unique().tolist())),
})
summary

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

## 5. Understanding the Variables

The processed dataset contains both demand information and contextual business variables. This is useful because retail demand is rarely explained by history alone.

Important variables include:

- `sales` as the forecasting target,
- `onpromotion` as a measure of promotional activity,
- `transactions` as a proxy for daily store traffic,
- `dcoilwtico` as an external time-varying economic signal,
- `holiday` and `national_holiday` as event indicators,
- calendar columns such as month, weekday, and weekend flags.

In [ ]:
feature_overview = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'null_count': [int(df[col].isna().sum()) for col in df.columns],
})
feature_overview

## 6. Exploratory Data Analysis

Before building forecasting models, I wanted to understand the basic sales behaviour of the selected families. The main questions guiding the EDA are:

- which family contributes the most sales,
- how much variability exists across families,
- whether there are visible weekly or monthly patterns,
- whether promotions and transactions are likely to matter.

In [ ]:
df.groupby('family')['sales'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)

In [ ]:
plt.figure(figsize=(12, 6))
for family, family_frame in df.groupby('family'):
    plt.plot(family_frame['date'], family_frame['sales'], label=family, linewidth=1.5)
plt.title('Daily Sales by Product Family for Store 1')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'daily_sales_by_family_notebook.png', dpi=200)
plt.show()

In [ ]:
monthly_sales = (
    df.assign(month_start=df['date'].dt.to_period('M').dt.to_timestamp())
      .groupby(['month_start', 'family'])['sales']
      .sum()
      .reset_index()
)

plt.figure(figsize=(12, 6))
for family, family_frame in monthly_sales.groupby('family'):
    plt.plot(family_frame['month_start'], family_frame['sales'], label=family, linewidth=1.7)
plt.title('Monthly Sales Trend by Product Family')
plt.xlabel('Month')
plt.ylabel('Monthly Total Sales')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
weekday_pattern = (
    df.groupby(['day_name', 'family'])['sales']
      .mean()
      .reset_index()
)
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_pattern['day_name'] = pd.Categorical(weekday_pattern['day_name'], categories=weekday_order, ordered=True)
weekday_pattern = weekday_pattern.sort_values(['day_name', 'family'])
weekday_pattern.head()

In [ ]:
weekday_pattern.pivot(index='day_name', columns='family', values='sales').plot(kind='bar', figsize=(12, 6))
plt.title('Average Sales by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('family')[['sales', 'onpromotion', 'transactions']].mean().round(2)

In [ ]:
df[['sales', 'onpromotion', 'transactions', 'dcoilwtico', 'holiday', 'national_holiday', 'is_weekend']].corr().round(2)

### Interpretation of the EDA

From the exploratory analysis, a few practical observations become clear:

- `GROCERY I` has the highest demand volume and also shows the largest spread.
- `BEVERAGES` is another strong family with visible day-to-day swings.
- `DAIRY` has lower average sales, but its variation is still meaningful enough for forecasting.
- Weekly effects, promotions, and store activity seem relevant, so it is reasonable to include both lag-based and business-context features in the modeling stage.

## 7. Feature Engineering

Retail time-series models often improve when they can see recent demand history. For that reason, I created lag features and rolling summary features. I also encoded a few categorical variables so they can be used numerically in a regression-style model.

In [ ]:
feature_df = df.copy()
feature_df['city_code'] = pd.Categorical(feature_df['city']).codes
feature_df['state_code'] = pd.Categorical(feature_df['state']).codes
feature_df['store_type_code'] = pd.Categorical(feature_df['type']).codes
feature_df['day_name_code'] = pd.Categorical(feature_df['day_name']).codes

for lag in [1, 7, 14, 28]:
    feature_df[f'lag_{lag}'] = feature_df.groupby('family')['sales'].shift(lag)

for window in [7, 14, 28]:
    shifted = feature_df.groupby('family')['sales'].shift(1)
    feature_df[f'rolling_mean_{window}'] = shifted.groupby(feature_df['family']).rolling(window).mean().reset_index(level=0, drop=True)
    feature_df[f'rolling_std_{window}'] = shifted.groupby(feature_df['family']).rolling(window).std().reset_index(level=0, drop=True)

for column in [c for c in feature_df.columns if c.startswith('rolling_std_')]:
    feature_df[column] = feature_df[column].fillna(0)

feature_df = feature_df.dropna().reset_index(drop=True)
feature_df.head()

In [ ]:
feature_df.shape

## 8. Train-Test Split Strategy

Because this is time-series data, a random split would leak future information into the training set. I therefore reserve the latest 90 days as the test set and use earlier data for training.

This creates a more realistic forecasting evaluation.

In [ ]:
cutoff = feature_df['date'].max() - pd.Timedelta(days=90)
train_df = feature_df[feature_df['date'] <= cutoff].copy()
test_df = feature_df[feature_df['date'] > cutoff].copy()

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))
print('Cutoff date:', cutoff.date())

## 9. Forecasting Models Considered

The original project plan mentioned a combination of classical and AI-based forecasting methods. In the current environment, I can compare the following models:

- **Seasonal naive** as a simple benchmark,
- **7-day moving average** as another short-term baseline,
- **Linear regression with lag features** as a structured machine-learning-style model,
- **ARIMA** as a classical time-series method,
- **Prophet** as an additive forecasting model.

LSTM is included at the pipeline level as an optional hook, but it depends on TensorFlow/Keras being available in the local environment.

In [ ]:
FEATURE_COLUMNS = [
    'onpromotion', 'item_count', 'perishable_share', 'city_code', 'state_code',
    'store_type_code', 'cluster', 'transactions', 'dcoilwtico', 'holiday',
    'national_holiday', 'holiday_event_count', 'national_event_count',
    'year', 'month', 'day', 'day_name_code', 'day_of_week', 'week_of_year',
    'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_14', 'rolling_std_28'
]

def mae(actual, predicted):
    return float(np.mean(np.abs(actual.to_numpy() - predicted.to_numpy())))

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual.to_numpy() - predicted.to_numpy()) ** 2)))

def mape(actual, predicted):
    actual_values = actual.to_numpy(dtype=float)
    predicted_values = predicted.to_numpy(dtype=float)
    denominator = np.where(actual_values == 0, 1, actual_values)
    return float(np.mean(np.abs((actual_values - predicted_values) / denominator)) * 100)

In [ ]:
def seasonal_naive_predict(train_frame, test_frame, season_lag=7):
    predictions = []
    ordered_test = test_frame.sort_values(['family', 'date'])
    for family, family_test in ordered_test.groupby('family'):
        family_train = train_frame[train_frame['family'] == family].sort_values('date')
        history = family_train['sales'].tolist()
        for _idx, row in family_test.iterrows():
            prediction = history[-season_lag] if len(history) >= season_lag else float(np.mean(history))
            predictions.append(max(prediction, 0.0))
            history.append(row['sales'])
    return pd.Series(predictions, index=ordered_test.index)

def moving_average_predict(train_frame, test_frame, window=7):
    predictions = []
    ordered_test = test_frame.sort_values(['family', 'date'])
    for family, family_test in ordered_test.groupby('family'):
        family_train = train_frame[train_frame['family'] == family].sort_values('date')
        history = family_train['sales'].tolist()
        for _idx, row in family_test.iterrows():
            values = history[-window:] if len(history) >= window else history
            prediction = float(np.mean(values))
            predictions.append(max(prediction, 0.0))
            history.append(row['sales'])
    return pd.Series(predictions, index=ordered_test.index)

def regression_predict(train_frame, test_frame):
    x_train = train_frame[FEATURE_COLUMNS].to_numpy(dtype=float)
    y_train = train_frame['sales'].to_numpy(dtype=float)
    train_design = np.hstack([np.ones((x_train.shape[0], 1)), x_train])
    coefficients, *_ = np.linalg.lstsq(train_design, y_train, rcond=None)

    ordered_test = test_frame.sort_values(['family', 'date'])
    x_test = ordered_test[FEATURE_COLUMNS].to_numpy(dtype=float)
    test_design = np.hstack([np.ones((x_test.shape[0], 1)), x_test])
    predictions = test_design @ coefficients
    return pd.Series(np.clip(predictions, a_min=0, a_max=None), index=ordered_test.index)

In [ ]:
def arima_predict(train_frame, test_frame):
    from statsmodels.tsa.arima.model import ARIMA

    predictions = []
    ordered_test = test_frame.sort_values(['family', 'date'])
    for family, family_test in ordered_test.groupby('family'):
        history = train_frame[train_frame['family'] == family].sort_values('date')['sales'].astype(float)
        fitted = ARIMA(history, order=(7, 1, 1)).fit()
        forecast = fitted.forecast(steps=len(family_test))
        predictions.extend(np.clip(forecast.to_numpy(), a_min=0, a_max=None).tolist())
    return pd.Series(predictions, index=ordered_test.index)

try:
    from prophet import Prophet
    prophet_available = True
except Exception:
    prophet_available = False

def prophet_predict(train_frame, test_frame):
    if not prophet_available:
        raise RuntimeError('Prophet is not available in the current environment.')

    predictions = []
    ordered_test = test_frame.sort_values(['family', 'date'])
    for family, family_test in ordered_test.groupby('family'):
        family_train = train_frame[train_frame['family'] == family].sort_values('date')
        prophet_train = family_train[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})
        model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        model.fit(prophet_train)
        future = family_test[['date']].rename(columns={'date': 'ds'})
        forecast = model.predict(future)
        predictions.extend(forecast['yhat'].clip(lower=0).tolist())
    return pd.Series(predictions, index=ordered_test.index)

## 10. Model Evaluation

I evaluate each model using three error metrics:

- **MAE** for average absolute error,
- **RMSE** for stronger penalty on large mistakes,
- **MAPE** for percentage-style interpretation.

For selection, I prioritize RMSE because large forecast errors can be especially costly in inventory planning.

In [ ]:
ordered_test = test_df.sort_values(['family', 'date']).copy()
actual = ordered_test['sales']

model_predictions = {
    'seasonal_naive': seasonal_naive_predict(train_df, ordered_test),
    'moving_average_7': moving_average_predict(train_df, ordered_test),
    'linear_regression_lags': regression_predict(train_df, ordered_test),
    'arima': arima_predict(train_df, ordered_test),
}

if prophet_available:
    model_predictions['prophet'] = prophet_predict(train_df, ordered_test)

results = []
best_model_name = None
best_predictions = None
best_model_rmse = float('inf')

for model_name, pred in model_predictions.items():
    aligned = pred.loc[ordered_test.index]
    current_rmse = rmse(actual, aligned)
    results.append({
        'model': model_name,
        'mae': round(mae(actual, aligned), 3),
        'rmse': round(current_rmse, 3),
        'mape': round(mape(actual, aligned), 3),
    })
    if current_rmse < best_model_rmse:
        best_model_rmse = current_rmse
        best_model_name = model_name
        best_predictions = aligned.copy()

metrics_df = pd.DataFrame(results).sort_values('rmse').reset_index(drop=True)
metrics_df

In [ ]:
availability_df = pd.DataFrame([
    {'model': 'seasonal_naive', 'available': True, 'note': 'Implemented locally as benchmark model.'},
    {'model': 'moving_average_7', 'available': True, 'note': 'Implemented locally as benchmark model.'},
    {'model': 'linear_regression_lags', 'available': True, 'note': 'Implemented locally using lag and rolling features.'},
    {'model': 'arima', 'available': True, 'note': 'Implemented using statsmodels ARIMA.'},
    {'model': 'prophet', 'available': prophet_available, 'note': 'Available only if Prophet is installed.'},
    {'model': 'lstm', 'available': False, 'note': 'TensorFlow/Keras is not available in the current environment.'},
])
availability_df

In [ ]:
ordered_test['predicted_sales'] = best_predictions.loc[ordered_test.index].to_numpy()
metrics_df.to_csv(TABLES_DIR / 'model_metrics_notebook.csv', index=False)
ordered_test.to_csv(TABLES_DIR / 'forecast_vs_actual_notebook.csv', index=False)
print('Best model:', best_model_name)

In [ ]:
families = ordered_test['family'].unique().tolist()
fig, axes = plt.subplots(len(families), 1, figsize=(12, 10), sharex=True)
if len(families) == 1:
    axes = [axes]

for axis, family in zip(axes, families):
    family_frame = ordered_test[ordered_test['family'] == family].sort_values('date')
    axis.plot(family_frame['date'], family_frame['sales'], label='Actual', linewidth=1.8)
    axis.plot(family_frame['date'], family_frame['predicted_sales'], label='Predicted', linewidth=1.4)
    axis.set_title(f'{family}: Actual vs Predicted Sales')
    axis.set_ylabel('Sales')
    axis.legend()

plt.xlabel('Date')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'actual_vs_predicted_notebook.png', dpi=200)
plt.show()

### What the Model Comparison Tells Me

This comparison is useful for two reasons. First, it tells me which model gives the lowest forecast error on the holdout period. Second, it helps me justify whether the added complexity of a model is worth it compared with simple baselines.

Even when a sophisticated model does not win, including it in the comparison strengthens the academic quality of the project.

## 11. Inventory Optimization and Adaptive Stock Logic

Forecasts become actionable when they are converted into stock rules. I therefore use the best-performing model to estimate:

- average daily forecast,
- forecast variability,
- safety stock,
- reorder point,
- recommended order quantity.

I then simulate inventory movement through the test period to see when reorder alerts would be triggered.

In [ ]:
lead_time_days = 7
review_period_days = 7
service_level_z = 1.65

inventory_rows = []
for family, family_frame in ordered_test.groupby('family'):
    avg_forecast = family_frame['predicted_sales'].mean()
    demand_std = family_frame['predicted_sales'].std(ddof=0)
    safety_stock = service_level_z * demand_std * np.sqrt(lead_time_days)
    reorder_point = avg_forecast * lead_time_days + safety_stock
    recommended_order_qty = avg_forecast * (lead_time_days + review_period_days) + safety_stock
    inventory_rows.append({
        'family': family,
        'avg_daily_forecast': round(float(avg_forecast), 2),
        'forecast_std_dev': round(float(demand_std), 2),
        'safety_stock': round(float(safety_stock), 2),
        'reorder_point': round(float(reorder_point), 2),
        'recommended_order_qty': round(float(recommended_order_qty), 2),
    })

inventory_df = pd.DataFrame(inventory_rows).sort_values('family').reset_index(drop=True)
inventory_df.to_csv(TABLES_DIR / 'inventory_recommendations_notebook.csv', index=False)
inventory_df

In [ ]:
simulation_rows = []
alert_rows = []

for family, family_frame in ordered_test.groupby('family'):
    family_policy = inventory_df[inventory_df['family'] == family].iloc[0]
    current_stock = float(family_policy['avg_daily_forecast'] * 10)
    reorder_point = float(family_policy['reorder_point'])
    order_qty = float(family_policy['recommended_order_qty'])
    open_orders = []

    for _, row in family_frame.sort_values('date').iterrows():
        current_date = row['date']
        actual_sales = float(row['sales'])
        predicted_sales = float(row['predicted_sales'])

        arrivals_today = sum(qty for arrival_date, qty in open_orders if arrival_date == current_date)
        current_stock += arrivals_today
        open_orders = [(arrival_date, qty) for arrival_date, qty in open_orders if arrival_date != current_date]

        stock_before_sales = current_stock
        fulfilled_sales = min(current_stock, actual_sales)
        stockout_units = max(actual_sales - current_stock, 0.0)
        current_stock = max(current_stock - actual_sales, 0.0)

        placed_order = 0.0
        if current_stock <= reorder_point:
            arrival_date = current_date + pd.Timedelta(days=7)
            open_orders.append((arrival_date, order_qty))
            placed_order = order_qty
            alert_rows.append({
                'date': current_date,
                'family': family,
                'alert_type': 'Reorder Triggered',
                'stock_after_sales': round(current_stock, 2),
                'reorder_point': round(reorder_point, 2),
                'recommended_order_qty': round(order_qty, 2),
            })

        simulation_rows.append({
            'date': current_date,
            'family': family,
            'actual_sales': round(actual_sales, 2),
            'predicted_sales': round(predicted_sales, 2),
            'stock_before_sales': round(stock_before_sales, 2),
            'fulfilled_sales': round(fulfilled_sales, 2),
            'stockout_units': round(stockout_units, 2),
            'stock_after_sales': round(current_stock, 2),
            'arrivals_today': round(arrivals_today, 2),
            'placed_order_qty': round(placed_order, 2),
        })

simulation_df = pd.DataFrame(simulation_rows)
alerts_df = pd.DataFrame(alert_rows)
simulation_df.to_csv(TABLES_DIR / 'inventory_simulation_notebook.csv', index=False)
alerts_df.to_csv(TABLES_DIR / 'stock_alerts_notebook.csv', index=False)
alerts_df.head(10)

In [ ]:
families = simulation_df['family'].unique().tolist()
fig, axes = plt.subplots(len(families), 1, figsize=(12, 10), sharex=True)
if len(families) == 1:
    axes = [axes]

for axis, family in zip(axes, families):
    family_frame = simulation_df[simulation_df['family'] == family].sort_values('date')
    axis.plot(family_frame['date'], family_frame['stock_after_sales'], label='Stock After Sales', linewidth=1.6)
    axis.bar(family_frame['date'], family_frame['stockout_units'], label='Stockout Units', alpha=0.35)
    axis.set_title(f'{family}: Simulated Inventory Position')
    axis.set_ylabel('Units')
    axis.legend()

plt.xlabel('Date')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'inventory_simulation_notebook.png', dpi=200)
plt.show()

## 12. Dashboard and Alert Outputs

One of the requirements of the project is to provide a visual decision-support output. In the code pipeline, I generate a dashboard-style HTML report along with stock alert tables.

That means the project now produces not only evaluation metrics, but also operational outputs that a manager could inspect:

- demand forecast comparison,
- inventory recommendation table,
- stock alert preview,
- inventory simulation chart.

In [ ]:
dashboard_file = REPORTS_DIR / 'dashboard.html'
dashboard_file.exists(), str(dashboard_file)

In [ ]:
alerts_df.groupby('family').size().rename('alert_count').reset_index() if not alerts_df.empty else pd.DataFrame()

## 13. Requirement Coverage Check

At this stage, the project covers the original task flow as follows:

- **Data collection and cleaning**: completed through the preprocessing script and merged dataset.
- **EDA**: completed through family-level trend, monthly pattern, weekday, and correlation analysis.
- **Forecasting model comparison**: completed with baseline models plus ARIMA and Prophet.
- **Inventory optimization**: completed through safety stock, reorder point, and order quantity logic.
- **Adaptive stock control**: represented through daily inventory simulation and reorder alerts.
- **Dashboard development**: covered through the generated HTML dashboard and visual outputs.

The only major item that remains environment-dependent is **LSTM**, because TensorFlow/Keras is not currently available.

## 14. Limitations and Honest Notes

It is important that the notebook stays academically honest.

- LSTM is not executed here because the required deep learning library is not available in the environment.
- The inventory simulation is a practical adaptive control mechanism, but it is not a full reinforcement learning implementation.
- The dashboard generated in the project is a lightweight HTML dashboard rather than a deployed web app.

These limitations do not invalidate the project, but they should be stated clearly in the report and viva.

## 15. Final Conclusion

This project demonstrates a complete end-to-end retail analytics workflow. Starting from raw data sources, I prepared a structured demand dataset, explored sales behaviour, compared multiple forecasting models, and used the best forecast to support stock planning decisions.

What makes the project meaningful is not just the prediction step, but the fact that the predictions are connected to inventory recommendations, simulated stock movements, and alert outputs. That is why the work fits well as an applied M.Sc. Data Science project.